In [ ]:
# ============================================================
# STANDALONE TOP-K EVALUATION — MATCHES YOUR TRAINED MODEL
# ============================================================

import os, gc, json, pickle, re
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

PHASE3_DIR = "/content/drive/MyDrive/Instacart/phase3_outputs_final"
PHASE4_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_light_singlecell"
MODEL_DIR  = f"{PHASE4_DIR}/Light_TimeAwareAttentionLSTM_ReLU"
BEST_CKPT  = f"{MODEL_DIR}/checkpoint_best.pt"

PRODUCT_VOCAB_SIZE = 25001
AISLE_VOCAB_SIZE = 135
DEPT_VOCAB_SIZE = 22
SAFE_MAX_DOW = 7
SAFE_MAX_HOUR = 24

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_batch(path):
    b = load_pickle(path)
    return {
        "Xp": torch.tensor(b["Xp"], dtype=torch.long, device=DEVICE),
        "Xa": torch.tensor(b["Xa"], dtype=torch.long, device=DEVICE),
        "Xd": torch.tensor(b["Xd"], dtype=torch.long, device=DEVICE),
        "Xdow": torch.tensor(b["Xdow"], dtype=torch.long, device=DEVICE),
        "Xhr": torch.tensor(b["Xhr"], dtype=torch.long, device=DEVICE),
        "Xdays": torch.tensor(b["Xdays"], dtype=torch.float32, device=DEVICE),
        "y": torch.tensor(b["y"], dtype=torch.long, device=DEVICE)
    }

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_outputs, mask=None):
        e = torch.tanh(self.attn(rnn_outputs))
        scores = self.score(e).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), rnn_outputs).squeeze(1)
        return context, weights

class TimeAwareAttentionRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.product_emb = nn.Embedding(PRODUCT_VOCAB_SIZE, 64, padding_idx=0)
        self.aisle_emb = nn.Embedding(AISLE_VOCAB_SIZE, 8, padding_idx=0)
        self.dept_emb = nn.Embedding(DEPT_VOCAB_SIZE, 4, padding_idx=0)
        self.dow_emb = nn.Embedding(SAFE_MAX_DOW, 4, padding_idx=0)
        self.hour_emb = nn.Embedding(SAFE_MAX_HOUR, 4, padding_idx=0)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, 4),
            nn.ReLU(),
            nn.Linear(4, 4)
        )

        self.recency_beta = nn.Parameter(torch.tensor(0.10, dtype=torch.float32))

        input_dim = 64 + 8 + 4 + 4 + 4 + 4 + 1

        self.rnn = nn.LSTM(
            input_size=input_dim,
            hidden_size=64,
            batch_first=True
        )

        self.attention = AttentionLayer(64)
        self.fc1 = nn.Linear(64, 64)
        self.dropout = nn.Dropout(0.2)
        self.fc_out = nn.Linear(64, PRODUCT_VOCAB_SIZE)

    def forward(self, Xp, Xa, Xd, Xdow, Xhr, Xdays):
        p_emb = self.product_emb(Xp)
        a_emb = self.aisle_emb(Xa)
        d_emb = self.dept_emb(Xd)
        dw_emb = self.dow_emb(Xdow)
        hr_emb = self.hour_emb(Xhr)

        xdays_log = torch.log1p(Xdays)
        xdays_norm = xdays_log / (xdays_log.max().detach() + 1e-8)

        gap_feature = xdays_norm.unsqueeze(-1)
        time_encoded = self.time_mlp(gap_feature)

        x = torch.cat(
            [p_emb, a_emb, d_emb, dw_emb, hr_emb, time_encoded, gap_feature],
            dim=-1
        )

        beta = torch.clamp(self.recency_beta, min=0.0)
        recency_weight = torch.exp(-beta * xdays_norm).unsqueeze(-1)
        x = x * recency_weight

        mask = (Xp != 0).long()

        rnn_out, _ = self.rnn(x)
        context, attn_weights = self.attention(rnn_out, mask=mask)

        h = self.fc1(context)
        h = F.relu(h)
        h = self.dropout(h)

        logits = self.fc_out(h)
        return logits, attn_weights

# ---------------- LOAD MODEL ----------------

model = TimeAwareAttentionRNN().to(DEVICE)
checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best checkpoint.")
print("Best epoch:", checkpoint.get("epoch"))
print("Best val F1:", checkpoint.get("best_val_f1"))

# ---------------- FIND REAL TEST BATCHES ----------------

all_files = sorted(glob(os.path.join(PHASE3_DIR, "*.pkl")))

test_files = [
    f for f in all_files
    if re.match(r".*test_batch_\d{4}_\d{8}_\d{6}\.pkl$", f)
]

print("Test batch files found:", len(test_files))

# ---------------- TOP-K EVALUATION ----------------

@torch.no_grad()
def eval_topk(files, k_values=[5, 10], inner_bs=64):
    model.eval()

    hits = {k: 0 for k in k_values}
    total = 0

    for file_idx, f in enumerate(files):
        batch = load_batch(f)
        n = batch["y"].shape[0]

        for start in range(0, n, inner_bs):
            end = min(start + inner_bs, n)

            Xp = batch["Xp"][start:end]
            Xa = batch["Xa"][start:end]
            Xd = batch["Xd"][start:end]
            Xdow = batch["Xdow"][start:end]
            Xhr = batch["Xhr"][start:end]
            Xdays = batch["Xdays"][start:end]
            y = batch["y"][start:end].view(-1, 1)

            logits, _ = model(Xp, Xa, Xd, Xdow, Xhr, Xdays)

            for k in k_values:
                topk = torch.topk(logits, k=k, dim=1).indices
                hits[k] += (topk == y).any(dim=1).sum().item()

            total += y.size(0)

            del Xp, Xa, Xd, Xdow, Xhr, Xdays, y, logits
            torch.cuda.empty_cache()

        del batch
        gc.collect()
        torch.cuda.empty_cache()

        if (file_idx + 1) % 25 == 0 or (file_idx + 1) == len(files):
            print(f"Processed {file_idx + 1}/{len(files)} test files")

    results = {"total_test_samples": total}

    for k in k_values:
        hit_rate = hits[k] / total
        results[f"hit_rate@{k}"] = hit_rate
        results[f"recall@{k}"] = hit_rate
        results[f"precision@{k}"] = hit_rate / k

    return results

topk_results = eval_topk(test_files, k_values=[5, 10], inner_bs=64)

result = {
    "model_name": "Light_TimeAwareAttentionLSTM_ReLU",
    "best_epoch": checkpoint.get("epoch"),
    "best_val_f1": checkpoint.get("best_val_f1"),
    **topk_results
}

topk_df = pd.DataFrame([result])
display(topk_df)

topk_csv = f"{PHASE4_DIR}/phase4_topk_only_results.csv"
topk_json = f"{PHASE4_DIR}/phase4_topk_only_results.json"

topk_df.to_csv(topk_csv, index=False)
save_json([result], topk_json)

print("Saved:")
print(topk_csv)
print(topk_json)

Device: cuda
Loaded best checkpoint.
Best epoch: 49
Best val F1: 0.013981539519816566
Test batch files found: 411
Processed 25/411 test files
Processed 50/411 test files
Processed 75/411 test files
Processed 100/411 test files
Processed 125/411 test files
Processed 150/411 test files
Processed 175/411 test files
Processed 200/411 test files
Processed 225/411 test files
Processed 250/411 test files
Processed 275/411 test files
Processed 300/411 test files
Processed 325/411 test files
Processed 350/411 test files
Processed 375/411 test files
Processed 400/411 test files
Processed 411/411 test files


,model_name,best_epoch,best_val_f1,total_test_samples,hit_rate@5,recall@5,precision@5,hit_rate@10,recall@10,precision@10
0,Light_TimeAwareAttentionLSTM_ReLU,49,0.013982,6348182,0.101161,0.101161,0.020232,0.146892,0.146892,0.014689


Saved:
/content/drive/MyDrive/Instacart/phase4_outputs_light_singlecell/phase4_topk_only_results.csv
/content/drive/MyDrive/Instacart/phase4_outputs_light_singlecell/phase4_topk_only_results.json
